IMPORT LIBRARY

In [17]:
import pickle
import numpy as np
import tensorflow as tf

from tensorflow.keras.layers import Layer
from tensorflow.keras.utils import register_keras_serializable

CUSTOM OBJECT LOAD MODEL

In [18]:
@register_keras_serializable()
class IngredientsImportanceLayer(Layer):

    def __init__(self, factor=1.2, **kwargs):
        super().__init__(**kwargs)
        self.factor = factor

    def call(self, inputs):
        return inputs * self.factor

    def get_config(self):
        config = super().get_config()
        config.update({
            "factor": self.factor
        })
        return config

LOAD MODEL

In [19]:
model = tf.keras.models.load_model(
    "../models/mlp_recipe_model.keras",
    custom_objects={
        "IngredientsImportanceLayer": IngredientsImportanceLayer
    },
    compile=False
)

print("Model berhasil dimuat")

Model berhasil dimuat


LOAD VECTORIZER

In [20]:
with open(
    "../models/tfidf_vectorizer.pkl",
    "rb"
) as f:
    vectorizer = pickle.load(f)

print("Vectorizer berhasil dimuat")

Vectorizer berhasil dimuat


LOAD LABEL ENCODER

In [21]:
with open(
    "../models/label_encoder.pkl",
    "rb"
) as f:
    label_encoder = pickle.load(f)

print("Label Encoder berhasil dimuat")

Label Encoder berhasil dimuat


INPUT BAHAN MANUAL

In [22]:
user_input = input(
    "Masukkan bahan resep, pisahkan dengan koma: "
)

user_text = user_input.lower()

print("Input bahan:")
print(user_text)

Input bahan:
sapi, bawang putih, jahe


PREDIKSI TOP 3 KATEGORI

In [23]:
X_input = vectorizer.transform(
    [user_text]
)

X_input = X_input.toarray().astype(np.float32)

prediction = model.predict(
    X_input,
    verbose=0
)

top_indices = np.argsort(
    prediction[0]
)[::-1][:3]

print("\nHASIL PREDIKSI TOP 3 KATEGORI:")

for rank, index in enumerate(top_indices, start=1):
    category = label_encoder.inverse_transform(
        [index]
    )[0]

    score = prediction[0][index] * 100

    print(
        f"{rank}. {category} - {score:.2f}%"
    )


HASIL PREDIKSI TOP 3 KATEGORI:
1. sapi - 99.92%
2. ayam - 0.04%
3. telur - 0.02%
